# YouTube オーディエンス・ネットワーク分析

「視聴者が他にどんなチャンネルを見ているか」を、視聴者 × チャンネルの2部グラフとして分析するプロジェクト。

**スコープ（合意事項）**
- 出力は**集計のみ**。個人単位の登録チャンネル一覧は計算後に保持しない。
- 特定個人の名指し・プロファイリングはしない。
- 視聴者像はジャンル・系統の傾向という粗い粒度に留める。
- コメント投稿者のIDは計算の中間データとしてのみメモリ上で扱い、Stage 4 で匿名化する。

**進捗**
- [x] Stage 1: Google Cloud で API キー取得
- [x] Stage 2: 環境構築・接続テスト
- [x] Stage 3: データ収集（コメント投稿者 → 公開 subscriptions、ページング対応で全件取得）
- [x] Stage 4: ネットワーク構築（匿名化2部グラフ・ノイズ除去）
- [x] Stage 5-7: 共起ネットワーク / Louvain クラスタ / パネル重なり・lift 親和性
- [ ] Stage 8: 可視化（ネットワーク図＋クラスタ色分け / 静的PNG・対話HTML）← いまここ
- [ ] Stage 9: クラスタ安定性（ブートストラップ ARI）
- [ ] Stage 10: Studio「視聴者」タブとの突き合わせ検証


## 共通設定

API キーは Codespaces secret（環境変数 `YOUTUBE_API_KEY`）から読み込む。ローカルなら `.env` でも同じコードで動く。

In [1]:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()  # ローカルなら .env を読む。Codespaces secret なら何もしない

API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "YOUTUBE_API_KEY が見つかりません。Codespaces secret か .env を確認してください。"
    )

youtube = build("youtube", "v3", developerKey=API_KEY)
print("クライアント作成 OK")

クライアント作成 OK


In [2]:
# 分析対象チャンネル（オーナー権限あり）。Stage 2 で確認済みのIDを使用。
CHANNEL_ID = "UCYo5jkYvVZY7wGItoRELILg"

# 接続テスト（公開情報の取得確認）
resp = youtube.channels().list(part="snippet,statistics", id=CHANNEL_ID).execute()
ch = resp["items"][0]
print("チャンネル名 :", ch["snippet"]["title"])
print("登録者数     :", ch["statistics"].get("subscriberCount", "非公開"))
print("動画数       :", ch["statistics"].get("videoCount"))

チャンネル名 : 内田博史【金持ちの習慣】
登録者数     : 213000
動画数       : 659


## Stage 3: データ収集

**手順**
1. 対象チャンネルのアップロード動画一覧を取得
2. 各動画のコメントから「コメント投稿者（＝視聴者の代理）」を集める
3. 各投稿者の**公開**登録チャンネルを取得し、`(視聴者, チャンネル)` のエッジを作る

**まずは小さくテスト**するためのパラメータが下のセル。`no_public_subs`（公開subが取れなかった人数）が多いのは想定どおり。取得できたエッジ数を見てから本番規模を決める。

In [ ]:
# --- テスト用パラメータ（最初はこの規模で動作確認）---
MAX_VIDEOS = 659                 # コメントを集める動画数（最新からN本）
MAX_COMMENTS_PER_VIDEO = 600   # 1動画あたり最大コメント数（100 = 1ページ）
MAX_SUBS_PER_USER = 1000         # 1ユーザーあたり取得する登録chの上限（50 = 1ページ）

In [4]:
# 1) アップロード動画の一覧から最新 MAX_VIDEOS 本の videoId を取得
ch_resp = youtube.channels().list(part="contentDetails", id=CHANNEL_ID).execute()
uploads_playlist = ch_resp["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

video_ids = []
req = youtube.playlistItems().list(
    part="contentDetails", playlistId=uploads_playlist, maxResults=50
)
while req is not None and len(video_ids) < MAX_VIDEOS:
    resp = req.execute()
    for item in resp["items"]:
        video_ids.append(item["contentDetails"]["videoId"])
        if len(video_ids) >= MAX_VIDEOS:
            break
    req = youtube.playlistItems().list_next(req, resp)

video_ids = video_ids[:MAX_VIDEOS]
print(f"対象動画: {len(video_ids)} 本")

対象動画: 423 本


In [6]:
# 2) 各動画のコメント投稿者を集める
from googleapiclient.errors import HttpError

commenter_ids = set()
skipped = 0
for vid in video_ids:
    collected = 0
    req = youtube.commentThreads().list(
        part="snippet", videoId=vid, maxResults=100, textFormat="plainText"
    )
    while req is not None and collected < MAX_COMMENTS_PER_VIDEO:
        try:
            resp = req.execute()
        except HttpError:
            # コメント無効・非公開などの動画はスキップ
            print(f"動画 {vid} はスキップ（コメント無効など）")
            skipped += 1
            break
        for item in resp["items"]:
            top = item["snippet"]["topLevelComment"]["snippet"]
            author = top.get("authorChannelId", {}).get("value")
            if author and author != CHANNEL_ID:   # チャンネル主自身は除く
                commenter_ids.add(author)
            collected += 1
            if collected >= MAX_COMMENTS_PER_VIDEO:
                break
        req = youtube.commentThreads().list_next(req, resp)

print(f"ユニークなコメント投稿者: {len(commenter_ids)} 人（スキップ動画: {skipped} 本）")

動画 lxpj9PyNV0A はスキップ（コメント無効など）
ユニークなコメント投稿者: 9497 人（スキップ動画: 1 本）


In [ ]:
# 3) 各投稿者の公開登録チャンネルを取得し、エッジを作る
#    edges は中間データ（メモリ上のみ）。Stage 4 で匿名化する。
#    subscriptions().list は1ページ最大50件。list_next でページングし、
#    1人あたり最大 MAX_SUBS_PER_USER 件まで取得する（旧版は先頭50件で打ち切られていた）。
from tqdm.auto import tqdm

edges = []          # (commenter_id, subscribed_channel_id, subscribed_channel_title)
no_public_subs = 0  # 登録が非公開 or 取得不可だった人数

for uid in tqdm(commenter_ids):
    collected = 0
    got_any = False
    req = youtube.subscriptions().list(
        part="snippet", channelId=uid, maxResults=50, order="alphabetical"
    )
    try:
        while req is not None and collected < MAX_SUBS_PER_USER:
            resp = req.execute()
            for it in resp.get("items", []):
                sub_id = it["snippet"]["resourceId"]["channelId"]
                sub_title = it["snippet"]["title"]
                edges.append((uid, sub_id, sub_title))
                got_any = True
                collected += 1
                if collected >= MAX_SUBS_PER_USER:
                    break
            req = youtube.subscriptions().list_next(req, resp)
    except Exception:
        pass               # 多くは「登録を非公開」。取れた分はそのまま使う。
    if not got_any:
        no_public_subs += 1

print(f"取得できたエッジ数      : {len(edges)}")
print(f"公開subが取れた人        : {len(commenter_ids) - no_public_subs} / {len(commenter_ids)}")
print(f"公開subが取れなかった人  : {no_public_subs}")


In [8]:
# 4) 集計プレビュー：視聴者層に多く共有されているチャンネル（＝Stage 5 の核の先取り）
import pandas as pd

df = pd.DataFrame(edges, columns=["commenter", "channel_id", "channel_title"])

# 自分自身のチャンネルは除外（当然みんな登録しているため）
df = df[df["channel_id"] != CHANNEL_ID]

# 1人が同じchを二重に数えないよう重複を除いてから、登録者数をカウント
unique_pairs = df.drop_duplicates(subset=["commenter", "channel_id"])
top = (unique_pairs.groupby(["channel_id", "channel_title"])["commenter"]
       .nunique().sort_values(ascending=False).head(20))

print("=== 視聴者が多く登録している他チャンネル Top20 ===")
print(top)

=== 視聴者が多く登録している他チャンネル Top20 ===
channel_id                channel_title                         
UC0PotgTwYxbOZuM9FbWKdcg  Paranoia_パラノイア【有益】                        460
UC-N7pA0rR1PrUBOfs2OA0oA  哲理学作家さとうみつろう『神さまとのおしゃべり』チャンネル             320
UC67Wr_9pA4I0glIxDt_Cpyw  両学長 リベラルアーツ大学                             267
UC4lN5sizuJraSHqy99xTy6Q  Naokiman Show                             205
UC1WkFVOCTPdY782AJ1PZ-JQ  Dr. Zion Kabasawa                         170
UC0FFHRF1mytLDhs6nxqGIQg  OTAKING / Toshio Okada                    152
UC8yHePe_RgUBE-waRWy6olw  PIVOT 公式チャンネル                             144
UC1bmnUH1ffc63zohkPvtXcQ  ブライトサイド | Bright Side Japan               133
UCFo4kqllbcQ4nV83WCyraiw  NAKATA UNIVERSITY                         123
UC-kF1uMFhIfvw6seHqDGwkg  アシタノワダイ                                   119
UC0yQ2h4gQXmVUFWZSqlMVOA  ひろゆき, hiroyuki                            111
UC9V4eJBNx_hOieGG51NZ6nA  フェルミ漫画大学                                  109
UCEixleMT76xDzoiEb9ZA7XA  本要約チャンネル【毎日1

### 結果の見方 / 次の判断

- `no_public_subs` が多くても正常（登録を非公開にしている人は取れない）。
- 取得できたエッジ数が少なすぎる場合は、`MAX_VIDEOS` と `MAX_COMMENTS_PER_VIDEO` を増やして視聴者の母数を広げる。
- このプレビューの Top チャンネルが「自分の視聴者が他に見ているチャンネル」の素データ。

ここまでの数字（コメント投稿者数 / 公開subが取れた人数 / エッジ数 / Top20）を確認できたら、本番規模を決めて **Stage 4（匿名化した2部グラフの構築）** に進む。

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm.auto import tqdm

# ---- セットアップ（このセル内で完結）----
load_dotenv()
API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise RuntimeError("YOUTUBE_API_KEY が見つかりません（Codespaces secret / .env を確認）")
youtube = build("youtube", "v3", developerKey=API_KEY)

CHANNEL_ID = "UCYo5jkYvVZY7wGItoRELILg"
CACHE = "edges_anon.csv"

MAX_VIDEOS = 659                # ← 前回の値に合わせる
MAX_COMMENTS_PER_VIDEO = 600    # ← 前回の値に合わせる
MAX_SUBS_PER_USER = 300         # ページをまたいだ「合計」取得上限（1ページ=最大50件）

# 旧データは1人50件で打ち切られていた（ページング未実装のため）。
# ページング修正後に一度だけ全件再収集するため True。完了後は False に戻すこと。
FORCE_RECOLLECT = True

# True のとき、既存キャッシュを .bak へ退避してから再収集する。
if FORCE_RECOLLECT:
    for path in (CACHE, "channel_statistics_cache.csv"):
        if os.path.exists(path):
            os.replace(path, path + ".bak")
            print(f"既存キャッシュを退避: {path} -> {path}.bak")


def fetch_public_subscriptions(channel_id, cap):
    """公開登録チャンネルをページングして最大 cap 件まで取得する。

    subscriptions().list は1ページ最大50件。order='alphabetical' は
    relevance と違いページングが途中で止まりにくく全件取得に向く。
    取得できた (sub_id, sub_title) のリストを返す（0件なら空）。
    """
    out = []
    req = youtube.subscriptions().list(
        part="snippet", channelId=channel_id,
        maxResults=50, order="alphabetical",
    )
    try:
        while req is not None and len(out) < cap:
            resp = req.execute()
            for it in resp.get("items", []):
                out.append((
                    it["snippet"]["resourceId"]["channelId"],
                    it["snippet"]["title"],
                ))
                if len(out) >= cap:
                    break
            req = youtube.subscriptions().list_next(req, resp)
    except Exception:
        # 登録非公開などで先頭ページから失敗することが多い。
        # 既に取れている分はそのまま使う。
        pass
    return out


# ---- 読み込み or 収集 ----
if os.path.exists(CACHE):
    df_anon = pd.read_csv(CACHE)
    print("キャッシュから読み込み（収集スキップ）:", df_anon.shape)
else:
    print("キャッシュなし → 収集を実行します（クォータ消費）")

    # 1) アップロード動画の最新 MAX_VIDEOS 本
    ch = youtube.channels().list(part="contentDetails", id=CHANNEL_ID).execute()
    uploads = ch["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
    video_ids = []
    req = youtube.playlistItems().list(part="contentDetails", playlistId=uploads, maxResults=50)
    while req is not None and len(video_ids) < MAX_VIDEOS:
        r = req.execute()
        video_ids += [i["contentDetails"]["videoId"] for i in r["items"]]
        req = youtube.playlistItems().list_next(req, r)
    video_ids = video_ids[:MAX_VIDEOS]
    print("対象動画:", len(video_ids))

    # 2) コメント投稿者（コメント無効動画はスキップ）
    commenter_ids = set()
    for vid in tqdm(video_ids, desc="comments"):
        collected = 0
        req = youtube.commentThreads().list(part="snippet", videoId=vid,
                                            maxResults=100, textFormat="plainText")
        while req is not None and collected < MAX_COMMENTS_PER_VIDEO:
            try:
                r = req.execute()
            except HttpError:
                break
            for it in r["items"]:
                a = it["snippet"]["topLevelComment"]["snippet"].get("authorChannelId", {}).get("value")
                if a and a != CHANNEL_ID:
                    commenter_ids.add(a)
                collected += 1
                if collected >= MAX_COMMENTS_PER_VIDEO:
                    break
            req = youtube.commentThreads().list_next(req, r)
    print("コメント投稿者:", len(commenter_ids))

    # 3) 公開subscriptions（ページングで全件 / 最大 MAX_SUBS_PER_USER 件）
    rows, no_pub = [], 0
    for uid in tqdm(commenter_ids, desc="subscriptions"):
        subs = fetch_public_subscriptions(uid, MAX_SUBS_PER_USER)
        if not subs:
            no_pub += 1
            continue
        for sub_id, sub_title in subs:
            rows.append((uid, sub_id, sub_title))
    print(f"公開subが取れた人: {len(commenter_ids)-no_pub}/{len(commenter_ids)}  エッジ: {len(rows)}")

    df = pd.DataFrame(rows, columns=["commenter", "channel_id", "channel_title"])

    # 4) 匿名化（対応表は即破棄）→ キャッシュ保存
    amap = {u: f"u{i}" for i, u in enumerate(df["commenter"].unique())}
    df_anon = df.copy()
    df_anon["viewer"] = df_anon["commenter"].map(amap)
    df_anon = df_anon.drop(columns=["commenter"])
    del amap
    df_anon = (df_anon[df_anon["channel_id"] != CHANNEL_ID]
               .drop_duplicates(subset=["viewer", "channel_id"]))
    df_anon.to_csv(CACHE, index=False)
    print("匿名エッジを保存:", df_anon.shape)


In [2]:
# Stage 4: ノイズ除去と分析対象の確定
# コメント投稿者かつ公開登録チャンネルを取得できた人を「観測パネル」と呼ぶ。
# これはチャンネル全体の視聴者母集団ではないため、以後の比率もこのパネル内の値として解釈する。

import numpy as np
import pandas as pd

N_OBSERVED_PANEL = df_anon["viewer"].nunique()

# 1人だけが登録しているチャンネルは、共起ネットワークの根拠として弱いため除外する。
MIN_VIEWERS = 3

# 残存チャンネルが1件だけの視聴者は、チャンネル間の共起を作れないため除外する。
MIN_CHANNELS_PER_VIEWER = 2


def first_nonempty(values):
    """チャンネル名の表記揺れに対し、最初の非空値を採用する。"""
    values = values.dropna()
    return values.iloc[0] if not values.empty else None


def summarize_channel_population(edges):
    """channel_id 単位で、観測パネル内の登録者数を集計する。"""
    return (
        edges.groupby("channel_id")
        .agg(
            channel_title=("channel_title", first_nonempty),
            n_viewers=("viewer", "nunique"),
        )
        .reset_index()
        .sort_values("n_viewers", ascending=False)
    )


# チャンネルと視聴者を交互に除外し、閾値を満たす部分グラフに収束させる。
# これにより、最終的な channel_pop と edges_f の母集団を一致させる。
edges_f = (
    df_anon[["viewer", "channel_id", "channel_title"]]
    .drop_duplicates(subset=["viewer", "channel_id"])
    .copy()
)

for _ in range(10):
    before_shape = edges_f.shape

    population = summarize_channel_population(edges_f)
    kept = set(
        population.loc[
            population["n_viewers"] >= MIN_VIEWERS,
            "channel_id",
        ]
    )
    edges_f = edges_f[edges_f["channel_id"].isin(kept)].copy()

    viewer_degree = (
        edges_f.groupby("viewer")["channel_id"]
        .nunique()
    )
    valid_viewers = set(
        viewer_degree[
            viewer_degree >= MIN_CHANNELS_PER_VIEWER
        ].index
    )
    edges_f = edges_f[edges_f["viewer"].isin(valid_viewers)].copy()

    if edges_f.shape == before_shape:
        break

channel_pop_all = summarize_channel_population(
    df_anon[["viewer", "channel_id", "channel_title"]]
    .drop_duplicates(subset=["viewer", "channel_id"])
)

channel_pop = summarize_channel_population(edges_f)
kept = set(channel_pop["channel_id"])

title_map = (
    channel_pop_all
    .set_index("channel_id")["channel_title"]
    .to_dict()
)

print(f"観測パネル人数: {N_OBSERVED_PANEL:,}")
print(f"全チャンネル数: {channel_pop_all.shape[0]:,}")
print(
    f"最終ネットワーク対象チャンネル数 "
    f"(登録者 {MIN_VIEWERS} 人以上): {len(kept):,}"
)
print(
    f"最終ネットワーク対象視聴者数 "
    f"(登録チャンネル {MIN_CHANNELS_PER_VIEWER} 件以上): "
    f"{edges_f['viewer'].nunique():,}"
)
print(f"最終ネットワーク用エッジ数: {len(edges_f):,}")


観測パネル人数: 1,527
全チャンネル数: 31,277
最終ネットワーク対象チャンネル数 (登録者 3 人以上): 4,217
最終ネットワーク対象視聴者数 (登録チャンネル 2 件以上): 1,497
最終ネットワーク用エッジ数: 34,866


In [4]:
# Stage 5: 登録チャンネル共起ネットワークの構築
# 単純な二部グラフ投影では、登録チャンネル数が多い視聴者1人が大量の結線を作る。
# そこで、各視聴者の寄与を 1 / (登録チャンネル数 - 1) に近づける重み付けを行う。

import networkx as nx
from scipy.sparse import csr_matrix, triu

MIN_SHARED_VIEWERS = 3
MIN_WEIGHTED_COSINE = 0.12

if edges_f.empty:
    raise ValueError("ネットワーク用エッジがありません。MIN_VIEWERS を下げてください。")

viewer_index = pd.Index(
    edges_f["viewer"].unique(),
    name="viewer",
)
channel_index = pd.Index(
    sorted(edges_f["channel_id"].unique()),
    name="channel_id",
)

row = viewer_index.get_indexer(edges_f["viewer"])
col = channel_index.get_indexer(edges_f["channel_id"])

# 行 = 視聴者、列 = チャンネルの二値行列
X = csr_matrix(
    (
        np.ones(len(edges_f), dtype=np.float32),
        (row, col),
    ),
    shape=(len(viewer_index), len(channel_index)),
)
X.sum_duplicates()

viewer_degree = np.asarray(X.sum(axis=1)).ravel()
valid_rows = viewer_degree > 1

X = X[valid_rows]
viewer_degree = viewer_degree[valid_rows]

# 各視聴者が多くのチャンネルを登録しているほど、1組あたりの寄与を下げる。
# X_weighted.T @ X_weighted の各共起には 1 / (degree - 1) が加わる。
viewer_weight = 1 / np.sqrt(viewer_degree - 1)
X_weighted = X.multiply(viewer_weight[:, None]).tocsr()

# raw_cooccurrence: 生の共通登録者数
# weighted_cooccurrence: ヘビーユーザーの影響を補正した共起
raw_cooccurrence = (X.T @ X).tocsr()
weighted_cooccurrence = (X_weighted.T @ X_weighted).tocsr()

upper = triu(raw_cooccurrence, k=1, format="coo")
raw_shared = upper.data.astype(int)

weighted_overlap = np.asarray(
    weighted_cooccurrence[upper.row, upper.col]
).reshape(-1)

weighted_support = np.asarray(
    X_weighted.power(2).sum(axis=0)
).reshape(-1)

denominator = np.sqrt(
    weighted_support[upper.row]
    * weighted_support[upper.col]
)

weighted_cosine = np.divide(
    weighted_overlap,
    denominator,
    out=np.zeros_like(weighted_overlap, dtype=float),
    where=denominator > 0,
)

# 「共通登録者が少ない偶然の接続」と「正規化しても弱い接続」を除く。
mask = (
    (raw_shared >= MIN_SHARED_VIEWERS)
    & (weighted_cosine >= MIN_WEIGHTED_COSINE)
)

edge_table = pd.DataFrame({
    "channel_id_a": channel_index[upper.row[mask]],
    "channel_id_b": channel_index[upper.col[mask]],
    "shared_viewers": raw_shared[mask],
    "fractional_overlap": weighted_overlap[mask],
    "weighted_cosine": weighted_cosine[mask],
})

G_ch = nx.Graph()
G_ch.add_edges_from(
    (
        row.channel_id_a,
        row.channel_id_b,
        {
            "weight": float(row.weighted_cosine),
            "shared_viewers": int(row.shared_viewers),
            "fractional_overlap": float(row.fractional_overlap),
        },
    )
    for row in edge_table.itertuples(index=False)
)

print(f"正規化後ネットワークのノード数: {G_ch.number_of_nodes():,}")
print(f"正規化後ネットワークのエッジ数: {G_ch.number_of_edges():,}")
print(f"共通登録者数の下限: {MIN_SHARED_VIEWERS}")
print(f"正規化コサイン類似度の下限: {MIN_WEIGHTED_COSINE}")


正規化後ネットワークのノード数: 1,679
正規化後ネットワークのエッジ数: 4,023
共通登録者数の下限: 3
正規化コサイン類似度の下限: 0.12


In [5]:
# Stage 5-補助: 正規化後ネットワークの中心チャンネルを確認
# ここでの重み付き次数は、単純な共通登録者数ではなく
# ヘビーユーザーの影響を補正したコサイン類似度の合計である。

if G_ch.number_of_nodes() == 0:
    raise ValueError(
        "フィルタ後のエッジがありません。"
        "MIN_SHARED_VIEWERS または MIN_WEIGHTED_COSINE を下げてください。"
    )

wdeg = sorted(
    G_ch.degree(weight="weight"),
    key=lambda item: item[1],
    reverse=True,
)[:15]

print("=== 正規化後ネットワークの中心チャンネル（重み付き次数 上位）===")
for channel_id, weighted_degree in wdeg:
    print(
        f"{title_map.get(channel_id, channel_id)[:34]:34s} "
        f"{weighted_degree:.4f}"
    )

edge_table.to_csv(
    "channel_network_edges_filtered.csv",
    index=False,
)


=== 正規化後ネットワークの中心チャンネル（重み付き次数 上位）===
哲理学作家さとうみつろう『神さまとのおしゃべり』チャンネル      18.3983
Paranoia_パラノイア【有益】                 11.6581
アシタノワダイ                            10.4838
OTAKING / Toshio Okada             8.3732
ブライトサイド | Bright Side Japan        8.3317
宇宙となかよし/Qさん                        7.3451
Nakano Hiroshi BooKtube Univ.      7.1176
IROHA TAROT                        7.0712
【FX初心者ch】 by ユーちぇる監督               6.8411
桜井美帆のオーラで開運チャンネル                   6.6499
なるためJAPAN                          6.6308
NAKATA UNIVERSITY                  6.5540
占い師けんけんTV                          6.4905
FXメガバンク – 今日から使えるトレード講座            6.4765
神結ちゃんねる 〜かみすちゃんねる〜                 6.4711


In [6]:
# Stage 6: Louvainクラスタリング
# クラスタ代表は「観測パネル内で単に人気」なチャンネルではなく、
# クラスタ内部での正規化された結びつきが強いチャンネルとして選ぶ。

from networkx.algorithms.community import louvain_communities

LOUVAIN_RESOLUTION = 1.0

communities = louvain_communities(
    G_ch,
    weight="weight",
    resolution=LOUVAIN_RESOLUTION,
    seed=42,
)

communities = sorted(
    communities,
    key=lambda comm: (-len(comm), sorted(comm)[0]),
)

pop = channel_pop.set_index("channel_id")

community_rows = []
channel_community_rows = []

for community_id, comm in enumerate(communities):
    subgraph = G_ch.subgraph(comm)
    internal_strength = dict(
        subgraph.degree(weight="weight")
    )

    representatives = sorted(
        comm,
        key=lambda channel_id: (
            internal_strength.get(channel_id, 0),
            pop["n_viewers"].get(channel_id, 0),
        ),
        reverse=True,
    )[:8]

    community_rows.append({
        "community_id": community_id,
        "n_channels": len(comm),
        "n_edges": subgraph.number_of_edges(),
        "internal_weight": round(
            sum(
                data["weight"]
                for _, _, data in subgraph.edges(data=True)
            ),
            4,
        ),
        "representative_channels": " / ".join(
            title_map.get(channel_id, channel_id)
            for channel_id in representatives
        ),
    })

    for channel_id in comm:
        channel_community_rows.append({
            "channel_id": channel_id,
            "channel": title_map.get(channel_id, channel_id),
            "community_id": community_id,
            "sample_viewers": int(
                pop["n_viewers"].get(channel_id, 0)
            ),
            "internal_strength": round(
                internal_strength.get(channel_id, 0),
                6,
            ),
        })

community_summary = (
    pd.DataFrame(community_rows)
    .sort_values(
        ["n_channels", "internal_weight"],
        ascending=False,
    )
)

channel_communities = (
    pd.DataFrame(channel_community_rows)
    .sort_values(
        ["community_id", "internal_strength"],
        ascending=[True, False],
    )
)

print(
    f"コミュニティ数: {len(communities)} "
    f"(resolution={LOUVAIN_RESOLUTION})"
)
print()
print(
    community_summary.head(20).to_string(
        index=False,
        max_colwidth=100,
    )
)

community_summary.to_csv(
    "channel_community_summary.csv",
    index=False,
)

channel_communities.to_csv(
    "channel_communities.csv",
    index=False,
)


コミュニティ数: 55 (resolution=1.0)

 community_id  n_channels  n_edges  internal_weight                                                                              representative_channels
            0         272      488          84.6540 NAKATA UNIVERSITY  / 両学長 リベラルアーツ大学 / Paranoia_パラノイア【有益】 / PIVOT 公式チャンネル / 学識サロン / メンタリスト DaiGo / ...
            1         235      597          99.8361 哲理学作家さとうみつろう『神さまとのおしゃべり』チャンネル / 宇宙となかよし/Qさん / 桜井美帆のオーラで開運チャンネル / 【完全覚醒の学校】YUKARI / プロ霊能力者チャンネル / ...
            2         219      516          97.2697 アシタノワダイ / なるためJAPAN / Nakano Hiroshi BooKtube Univ. / 占い師けんけんTV / カピバラチャンネル capybarachannel / Hai...
            3         127      186          36.0901 井川意高が熔ける日本を斬る / OTAKING / Toshio Okada / シュン@億マーケ / ひろゆき, hiroyuki / ひろゆきの部屋【ひろゆき, hiroyuki】切り抜き ...
            4         107      158          33.6888 Naokiman Show / NMS STUDIO / Dharma Talk through Scary Stories / TOLAND VLOG / ウマヅラのお茶の間 / ねずみ / ...
            5          87      123          24.6049 

In [7]:
# Stage 7: 観測パネル内の重なりを算出
# これは「親和性」の確定値ではない。
# 比較対象チャンネル群がないため、ここでは
# 「コメント投稿者かつ公開登録取得者の観測パネル内での重なり」を出す。

from pathlib import Path
from datetime import datetime, timezone
from math import sqrt

MIN_OBSERVED_COMMENTERS = 5
STATS_CACHE_FILE = Path("channel_statistics_cache.csv")

# True にすると、キャッシュ済みのチャンネルも再取得する。
REFRESH_CHANNEL_STATS = False


def wilson_lower_bound(successes, total, z=1.96):
    """少数サンプルの偶然の上振れを抑える95% Wilson下限。"""
    if total == 0:
        return 0.0

    p = successes / total
    denominator = 1 + (z ** 2 / total)
    center = p + (z ** 2 / (2 * total))
    margin = z * sqrt(
        (p * (1 - p) / total)
        + (z ** 2 / (4 * total ** 2))
    )
    return max(0.0, (center - margin) / denominator)


candidate_pop = (
    channel_pop[
        channel_pop["n_viewers"] >= MIN_OBSERVED_COMMENTERS
    ]
    .copy()
)

channel_ids = candidate_pop["channel_id"].tolist()

# 取得済み統計は再利用し、不要な API コールを避ける。
if STATS_CACHE_FILE.exists():
    stats_cache = pd.read_csv(
        STATS_CACHE_FILE,
        dtype={"channel_id": str},
    )
else:
    stats_cache = pd.DataFrame(columns=[
        "channel_id",
        "total_subs",
        "subscriber_count_hidden",
        "retrieved_at",
    ])

if not stats_cache.empty:
    stats_cache["channel_id"] = (
        stats_cache["channel_id"]
        .astype(str)
    )

cached_ids = (
    set(stats_cache["channel_id"])
    if not stats_cache.empty
    else set()
)

ids_to_fetch = (
    channel_ids
    if REFRESH_CHANNEL_STATS
    else [
        channel_id
        for channel_id in channel_ids
        if channel_id not in cached_ids
    ]
)

new_rows = []

for i in range(0, len(ids_to_fetch), 50):
    batch = ids_to_fetch[i:i + 50]

    response = youtube.channels().list(
        part="statistics",
        id=",".join(batch),
    ).execute()

    returned_ids = set()

    for item in response.get("items", []):
        channel_id = item["id"]
        statistics = item.get("statistics", {})
        subscriber_count = statistics.get("subscriberCount")

        new_rows.append({
            "channel_id": channel_id,
            "total_subs": (
                int(subscriber_count)
                if subscriber_count is not None
                else None
            ),
            "subscriber_count_hidden": bool(
                statistics.get(
                    "hiddenSubscriberCount",
                    False,
                )
            ),
            "retrieved_at": datetime.now(
                timezone.utc
            ).isoformat(),
        })
        returned_ids.add(channel_id)

    # 応答に含まれないIDも記録し、次回も同じIDを無限に取りに行かない。
    for channel_id in set(batch) - returned_ids:
        new_rows.append({
            "channel_id": channel_id,
            "total_subs": None,
            "subscriber_count_hidden": None,
            "retrieved_at": datetime.now(
                timezone.utc
            ).isoformat(),
        })

if new_rows:
    stats_cache = pd.concat(
        [stats_cache, pd.DataFrame(new_rows)],
        ignore_index=True,
    )
    stats_cache = (
        stats_cache
        .drop_duplicates(
            subset=["channel_id"],
            keep="last",
        )
    )

stats_cache.to_csv(
    STATS_CACHE_FILE,
    index=False,
)

panel_overlap_df = candidate_pop.merge(
    stats_cache[
        [
            "channel_id",
            "total_subs",
            "subscriber_count_hidden",
            "retrieved_at",
        ]
    ],
    on="channel_id",
    how="left",
)

panel_overlap_df = panel_overlap_df.rename(columns={
    "channel_title": "channel",
    "n_viewers": "observed_commenters",
})

panel_overlap_df["observed_panel_size"] = N_OBSERVED_PANEL
panel_overlap_df["sample_share"] = (
    panel_overlap_df["observed_commenters"]
    / panel_overlap_df["observed_panel_size"]
)

# 主ランキング: 観測パネル内で安定して多く確認できるチャンネル。
panel_overlap_df["sample_share_lcb95"] = (
    panel_overlap_df["observed_commenters"]
    .apply(
        lambda n: wilson_lower_bound(
            n,
            N_OBSERVED_PANEL,
        )
    )
)

panel_overlap_df["total_subs"] = pd.to_numeric(
    panel_overlap_df["total_subs"],
    errors="coerce",
)

# 補助ランキング: 小規模チャンネル探索用。
# 比較対象がないため、これを「親和性」として結論づけない。
panel_overlap_df["penetration_per_1M_subs"] = (
    panel_overlap_df["observed_commenters"]
    / panel_overlap_df["total_subs"]
    * 1_000_000
)

panel_overlap_df.loc[
    panel_overlap_df["total_subs"].isna()
    | (panel_overlap_df["total_subs"] <= 0),
    "penetration_per_1M_subs",
] = np.nan

main_columns = [
    "channel_id",
    "channel",
    "observed_commenters",
    "observed_panel_size",
    "sample_share",
    "sample_share_lcb95",
    "total_subs",
    "subscriber_count_hidden",
    "retrieved_at",
]

panel_overlap_df = panel_overlap_df.sort_values(
    [
        "sample_share_lcb95",
        "observed_commenters",
    ],
    ascending=False,
)

print(
    "=== 主ランキング: 観測パネル内での登録割合 "
    "（Wilson下限95%）==="
)
print(
    panel_overlap_df[
        main_columns
    ]
    .head(30)
    .to_string(index=False)
)

print()
print(
    "=== 補助ランキング: 小規模チャンネル探索用 "
    "（主指標ではない）==="
)
print(
    panel_overlap_df[
        panel_overlap_df["penetration_per_1M_subs"].notna()
    ]
    .sort_values(
        [
            "penetration_per_1M_subs",
            "observed_commenters",
        ],
        ascending=False,
    )[
        main_columns
        + ["penetration_per_1M_subs"]
    ]
    .head(30)
    .to_string(index=False)
)

panel_overlap_df.to_csv(
    "channel_panel_overlap_metrics.csv",
    index=False,
)


=== 主ランキング: 観測パネル内での登録割合 （Wilson下限95%）===
              channel_id                                channel  observed_commenters  observed_panel_size  sample_share  sample_share_lcb95  total_subs subscriber_count_hidden                     retrieved_at
UC0PotgTwYxbOZuM9FbWKdcg                     Paranoia_パラノイア【有益】                  470                 1527      0.307793            0.285148      398000                   False 2026-06-21T09:38:52.524786+00:00
UC-N7pA0rR1PrUBOfs2OA0oA          哲理学作家さとうみつろう『神さまとのおしゃべり』チャンネル                  329                 1527      0.215455            0.195561      752000                   False 2026-06-21T09:38:52.524703+00:00
UC67Wr_9pA4I0glIxDt_Cpyw                          両学長 リベラルアーツ大学                  269                 1527      0.176162            0.157874     9820000                   False 2026-06-21T09:38:52.524673+00:00
UC4lN5sizuJraSHqy99xTy6Q                          Naokiman Show                  209                 1527      0.136870   

In [ ]:
# Stage 7b: 親和性 lift（汎用人気 vs 自視聴者“固有”の親和性を分離）
# sample_share は「公開登録を持つコメント投稿者の間で globally 人気」を並べるだけで、
# 自視聴者に固有の親和性とは区別できない。チャンネル規模（登録者総数）で割り、
# 「サイズから期待される登録率に対し、どれだけ過剰/過少に登録されているか」を出す。

lift_df = panel_overlap_df.copy()

# 規模が分かる行のみで baseline を作る（hidden / 欠損 / 0 は除外）。
valid = lift_df["total_subs"].notna() & (lift_df["total_subs"] > 0)

size_share = lift_df.loc[valid, "total_subs"] / lift_df.loc[valid, "total_subs"].sum()
panel_share = (
    lift_df.loc[valid, "observed_commenters"]
    / lift_df.loc[valid, "observed_commenters"].sum()
)
raw_lift = panel_share / size_share

# 中央値=1 へ自己校正。>1 = サイズの割に自視聴者へ過剰登録（固有親和性）、
# ≈1 = 汎用人気、<1 = 過少。外部ベースライン無しで相対比較できる。
median_lift = raw_lift.median()
lift_df["affinity_lift"] = np.nan
lift_df.loc[valid, "affinity_lift"] = raw_lift / median_lift

# 少数サンプルの偶然の上振れを避けるため、観測者数の下限を設けて表示する。
MIN_COMMENTERS_FOR_LIFT = 10

ranked = (
    lift_df[lift_df["observed_commenters"] >= MIN_COMMENTERS_FOR_LIFT]
    .dropna(subset=["affinity_lift"])
    .sort_values("affinity_lift", ascending=False)
)

cols = ["channel", "observed_commenters", "sample_share", "total_subs", "affinity_lift"]

print(f"=== 親和性 lift Top30（中央値=1.0 に校正 / 観測者{MIN_COMMENTERS_FOR_LIFT}人以上）===")
print("  lift>1 = 規模の割に自視聴者へ過剰登録（固有親和性が高い）")
print(ranked[cols].head(30).to_string(index=False))

print()
print("=== 参考: 大型だが lift は平凡（汎用人気寄り / lift 低い順）===")
print(ranked[cols].tail(10).to_string(index=False))

panel_overlap_df = lift_df
panel_overlap_df.to_csv("channel_panel_overlap_metrics.csv", index=False)


## Stage 8: ネットワーク図 + クラスタ色分けの可視化

`channel_network_edges_filtered.csv`（重み付き共起エッジ）と
`channel_communities.csv`（Louvain クラスタ割当）から自己完結で再構築する。
ノイズになる小クラスタ（チャンネル数 < `MIN_COMMUNITY_SIZE`）は除外し、
最大連結成分のみを描画する。

- ノード色 = コミュニティ、ノードサイズ = クラスタ内部強度
- ラベルは各クラスタのハブ（内部強度上位）のみ
- 出力: `channel_network_visualization.png`（静的）/ `channel_network_interactive.html`（対話）

日本語ラベルの文字化け回避のため `japanize-matplotlib` を使う
（未導入なら `pip install japanize-matplotlib`）。


In [ ]:
# Stage 8a: 静的ネットワーク図（matplotlib / クラスタ色分け）
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl

# 日本語フォント設定（japanize-matplotlib が無くてもシステムのCJKフォントへフォールバック）
def _setup_japanese_font():
    try:
        import japanize_matplotlib  # noqa: F401
        return "japanize-matplotlib"
    except Exception:
        pass
    import matplotlib.font_manager as fm
    candidates = [
        "Noto Sans CJK JP", "Noto Sans JP", "IPAexGothic", "IPAGothic",
        "IPAPGothic", "TakaoPGothic", "VL PGothic", "Source Han Sans JP",
        "WenQuanYi Zen Hei", "Yu Gothic", "Meiryo", "Hiragino Sans",
    ]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            # networkx のラベルは font_family='sans-serif' を使うため、
            # sans-serif の先頭に CJK フォントを差し込む（family も sans-serif に）。
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [name] + [
                f for f in plt.rcParams.get("font.sans-serif", []) if f != name
            ]
            plt.rcParams["axes.unicode_minus"] = False
            return name
    print("日本語フォントが見つかりません。ラベルが文字化けする場合は "
          "pip install japanize-matplotlib を実行してください。")
    return None


print("使用フォント:", _setup_japanese_font())

MIN_COMMUNITY_SIZE = 5   # これ未満のクラスタはノイズとして描画から除外

edges = pd.read_csv("channel_network_edges_filtered.csv")
comm = pd.read_csv("channel_communities.csv")

# 小クラスタ除去
sizes = comm.groupby("community_id").size()
big_comms = set(sizes[sizes >= MIN_COMMUNITY_SIZE].index)
comm_big = comm[comm["community_id"].isin(big_comms)].copy()
keep = set(comm_big["channel_id"])

G = nx.Graph()
for r in edges.itertuples(index=False):
    if r.channel_id_a in keep and r.channel_id_b in keep:
        G.add_edge(r.channel_id_a, r.channel_id_b, weight=float(r.weighted_cosine))

# 最大連結成分のみ
if G.number_of_nodes():
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()

meta = comm_big.set_index("channel_id")
node_comm = {n: int(meta.loc[n, "community_id"]) for n in G.nodes}
node_str = {n: float(meta.loc[n, "internal_strength"]) for n in G.nodes}
node_title = {n: str(meta.loc[n, "channel"]) for n in G.nodes}

comm_ids = sorted(set(node_comm.values()))
palette = plt.cm.tab20(np.linspace(0, 1, max(len(comm_ids), 1)))
color_of = {c: palette[i % len(palette)] for i, c in enumerate(comm_ids)}
node_colors = [color_of[node_comm[n]] for n in G.nodes]

s = np.array([node_str[n] for n in G.nodes], dtype=float)
rng = np.ptp(s) or 1.0
node_sizes = 30 + 320 * (s - s.min()) / rng

pos = nx.spring_layout(G, weight="weight", seed=42, k=0.30, iterations=200)

fig, ax = plt.subplots(figsize=(18, 18))
nx.draw_networkx_edges(G, pos, alpha=0.06, width=0.5, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                       linewidths=0, ax=ax)

# 各クラスタのハブ（内部強度上位2件）だけラベル表示
labels = {}
for c in comm_ids:
    members = [n for n in G.nodes if node_comm[n] == c]
    for n in sorted(members, key=lambda x: node_str[x], reverse=True)[:2]:
        labels[n] = node_title[n]
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, ax=ax)

# 凡例: クラスタ代表（最大内部強度のチャンネル名）
handles = []
for c in comm_ids:
    members = [n for n in G.nodes if node_comm[n] == c]
    rep = max(members, key=lambda x: node_str[x])
    handles.append(mpl.lines.Line2D(
        [], [], marker="o", linestyle="", markersize=8,
        markerfacecolor=color_of[c], markeredgewidth=0,
        label=f"C{c}: {node_title[rep][:18]}",
    ))
ax.legend(handles=handles, loc="upper left", fontsize=8,
          framealpha=0.9, ncol=1, title="クラスタ代表")

ax.set_title(
    f"YouTube 視聴者の登録チャンネル共起ネットワーク "
    f"（{G.number_of_nodes()} ch / {len(comm_ids)} クラスタ）",
    fontsize=14,
)
ax.axis("off")
plt.tight_layout()
plt.savefig("channel_network_visualization.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: channel_network_visualization.png "
      f"({G.number_of_nodes()} nodes / {G.number_of_edges()} edges)")


In [ ]:
# Stage 8b: 対話的ネットワーク図（pyvis / 同じ G・同色を流用）
# ノードをドラッグ・ズーム・ホバーで確認できる HTML を書き出す。
try:
    from pyvis.network import Network

    net = Network(height="800px", width="100%", bgcolor="#ffffff",
                  font_color="#222222", notebook=False, cdn_resources="in_line")
    net.barnes_hut()

    for n in G.nodes:
        net.add_node(
            n,
            label=node_title[n],
            title=f"{node_title[n]} / クラスタ C{node_comm[n]} / 内部強度 {node_str[n]:.3f}",
            color=mpl.colors.to_hex(color_of[node_comm[n]]),
            value=float(node_str[n]),
        )
    for a, b, d in G.edges(data=True):
        net.add_edge(a, b, value=float(d["weight"]))

    net.save_graph("channel_network_interactive.html")
    print("保存: channel_network_interactive.html（ブラウザで開く）")
except Exception as e:
    print("pyvis 未導入のためスキップ:", e)
    print("対話版が必要なら pip install pyvis を実行してください。")


## Stage 9: クラスタ安定性チェック（サブサンプリング ARI）

得られたクラスタが「偶然の産物」でなく頑健かを確認する。視聴者を非復元で
`SUBSAMPLE_FRAC`（既定80%）だけ抜き出して共起ネットワーク → Louvain をやり直し、
毎回の分割を参照分割（`channel_communities.csv`）と Adjusted Rand Index (ARI) で比較する。

- 復元抽出（bootstrap）だと同一視聴者が複製され、`MIN_SHARED_VIEWERS`（本来は異なる3人）の
  閾値を1人で通してしまうため、ここでは**非復元サブサンプリング**を使う。
- **コア（内部強度上位）ARI** と **全ノード ARI** を分けて報告する。クラスタを定義するのは
  ハブ＝コアであり、周辺の弱いノードは抽出のゆらぎで入れ替わりやすい。ARI が高い（1に近い）
  ほど安定。


In [ ]:
# Stage 9: サブサンプリングによるクラスタ安定性（Adjusted Rand Index）
import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix, triu
from networkx.algorithms.community import louvain_communities

# Stage 4-6 と同じ閾値（再現性のため明示）
MIN_VIEWERS = 3
MIN_CHANNELS_PER_VIEWER = 2
MIN_SHARED_VIEWERS = 3
MIN_WEIGHTED_COSINE = 0.12

# 視聴者を非復元で抽出する割合と試行回数。
SUBSAMPLE_FRAC = 0.8
N_RESAMPLES = 20
# クラスタを定義するハブ（内部強度上位）をコアとし、その安定性を本指標にする。
CORE_TOP_N = 200


def filter_bipartite(edges_bip):
    """Stage 4 と同じく、チャンネルと視聴者を交互に間引いて収束させる。"""
    e = edges_bip.drop_duplicates(subset=["viewer", "channel_id"]).copy()
    for _ in range(10):
        before = e.shape
        cv = e.groupby("channel_id")["viewer"].nunique()
        e = e[e["channel_id"].isin(cv[cv >= MIN_VIEWERS].index)]
        vd = e.groupby("viewer")["channel_id"].nunique()
        e = e[e["viewer"].isin(vd[vd >= MIN_CHANNELS_PER_VIEWER].index)]
        if e.shape == before:
            break
    return e


def build_partition(edges_bip):
    """2部エッジ → weighted-cosine 共起ネットワーク → Louvain 分割（dict）。"""
    e = filter_bipartite(edges_bip)
    if e.empty:
        return {}
    vidx = pd.Index(e["viewer"].unique())
    cidx = pd.Index(sorted(e["channel_id"].unique()))
    X = csr_matrix(
        (np.ones(len(e), np.float32),
         (vidx.get_indexer(e["viewer"]), cidx.get_indexer(e["channel_id"]))),
        shape=(len(vidx), len(cidx)),
    )
    X.sum_duplicates()
    deg = np.asarray(X.sum(axis=1)).ravel()
    X = X[deg > 1]
    deg = deg[deg > 1]
    Xw = X.multiply((1 / np.sqrt(deg - 1))[:, None]).tocsr()
    raw = (X.T @ X).tocsr()
    up = triu(raw, k=1, format="coo")
    shared = up.data.astype(int)
    wco = (Xw.T @ Xw).tocsr()
    overlap = np.asarray(wco[up.row, up.col]).ravel()
    supp = np.asarray(Xw.power(2).sum(axis=0)).ravel()
    denom = np.sqrt(supp[up.row] * supp[up.col])
    cos = np.divide(overlap, denom, out=np.zeros_like(overlap, float), where=denom > 0)
    m = (shared >= MIN_SHARED_VIEWERS) & (cos >= MIN_WEIGHTED_COSINE)
    G = nx.Graph()
    G.add_weighted_edges_from(zip(cidx[up.row[m]], cidx[up.col[m]], cos[m]))
    if G.number_of_nodes() == 0:
        return {}
    parts = louvain_communities(G, weight="weight", resolution=1.0, seed=42)
    return {ch: i for i, p in enumerate(parts) for ch in p}


def adjusted_rand_index(labels_a, labels_b):
    """共通ノード上の ARI（ペア計数による自前実装、sklearn 不使用）。"""
    common = sorted(set(labels_a) & set(labels_b))
    if len(common) < 2:
        return float("nan")
    a = pd.Series([labels_a[c] for c in common])
    b = pd.Series([labels_b[c] for c in common])
    ct = pd.crosstab(a, b).values.astype(float)
    comb2 = lambda x: x * (x - 1) / 2
    sum_ij = comb2(ct).sum()
    sum_a = comb2(ct.sum(axis=1)).sum()
    sum_b = comb2(ct.sum(axis=0)).sum()
    n = comb2(ct.sum())
    expected = sum_a * sum_b / n
    max_index = (sum_a + sum_b) / 2
    if max_index == expected:
        return 1.0
    return (sum_ij - expected) / (max_index - expected)


# 参照分割（本番の結果）と、その内部強度上位を「コア」とする
ref = pd.read_csv("channel_communities.csv")
ref_partition = dict(zip(ref["channel_id"], ref["community_id"]))
core = set(
    ref.sort_values("internal_strength", ascending=False)["channel_id"].head(CORE_TOP_N)
)
ref_core = {k: v for k, v in ref_partition.items() if k in core}

df_anon = pd.read_csv("edges_anon.csv")
base = df_anon[["viewer", "channel_id"]].drop_duplicates()
viewers = base["viewer"].unique()

rng = np.random.default_rng(42)
glob_aris, core_aris = [], []
for _ in range(N_RESAMPLES):
    take = rng.choice(viewers, size=int(len(viewers) * SUBSAMPLE_FRAC), replace=False)
    sub = base[base["viewer"].isin(set(take))]
    part = build_partition(sub)
    glob_aris.append(adjusted_rand_index(ref_partition, part))
    core_aris.append(adjusted_rand_index(ref_core, part))

glob_aris = np.array([a for a in glob_aris if not np.isnan(a)])
core_aris = np.array([a for a in core_aris if not np.isnan(a)])

print(f"視聴者 {int(SUBSAMPLE_FRAC * 100)}% サブサンプリング × {len(glob_aris)} 回")
print(f"  コア(内部強度上位{CORE_TOP_N}) ARI : 平均 {core_aris.mean():.3f}  "
      f"(min {core_aris.min():.3f} / max {core_aris.max():.3f})")
print(f"  全ノード ARI              : 平均 {glob_aris.mean():.3f}  "
      f"(min {glob_aris.min():.3f} / max {glob_aris.max():.3f})")
print()
print("  目安: 0.7+ で頑健、0.5前後で中程度、0.3未満は不安定")
print("  解釈: コアのジャンル構造が安定なら、全体が低くても主因は周辺ノードの揺らぎ。")


## Stage 10: Studio「視聴者」タブとの突き合わせ検証（オーナー権限）

重要: YouTube Analytics/Reporting API には「視聴者が他にどのチャンネルを見ているか」は
含まれない。この重なりデータは **YouTube Studio の「視聴者」タブ UI にしか無く**、
本分析（コメント投稿者の公開登録からの推定）の唯一の地上検証になる。

Studio →「アナリティクス」→「視聴者」→「このチャンネルの視聴者が視聴している
他のチャンネル」に出るチャンネル名を下のリストへ貼り付けて実行する。


In [ ]:
# Stage 10: Studio「視聴者」タブの一覧と本分析の上位を突き合わせる
import pandas as pd

# ↓ Studio の「視聴者が視聴している他のチャンネル」をそのまま貼り付け（1行1チャンネル名）
STUDIO_OTHER_CHANNELS = [
    # "両学長 リベラルアーツ大学",
    # "NAKATA UNIVERSITY",
]

TOP_N = 30
po = pd.read_csv("channel_panel_overlap_metrics.csv")

ours_share = (
    po.sort_values("sample_share_lcb95", ascending=False)["channel"].head(TOP_N).tolist()
)
ours_lift = (
    po.dropna(subset=["affinity_lift"]).sort_values("affinity_lift", ascending=False)["channel"].head(TOP_N).tolist()
    if "affinity_lift" in po.columns else []
)


def norm(s):
    return "".join(str(s).split()).lower()


if not STUDIO_OTHER_CHANNELS:
    print("STUDIO_OTHER_CHANNELS が空です。Studio の一覧を貼り付けて再実行してください。")
else:
    studio = {norm(s) for s in STUDIO_OTHER_CHANNELS}

    def report(name, ranked):
        hit = [c for c in ranked if norm(c) in studio]
        miss_ours = [c for c in ranked if norm(c) not in studio]
        print(f"=== {name}（Top{TOP_N}）vs Studio 一覧（{len(studio)}件）===")
        print(f"一致: {len(hit)} 件 -> {hit}")
        print(f"本分析は上位だが Studio に無い: {miss_ours[:15]}")
        print()

    report("sample_share ランキング", ours_share)
    if ours_lift:
        report("affinity_lift ランキング", ours_lift)

    matched = {norm(c) for c in ours_share} | {norm(c) for c in ours_lift}
    studio_only = [s for s in STUDIO_OTHER_CHANNELS if norm(s) not in matched]
    print("Studio にあるが本分析の上位に無い（要確認）:", studio_only[:15])
    print("注: 名前の表記揺れで取りこぼす場合あり。必要なら手で対応付けを。")
